# Plot BIAS CORRECTED OSDMA8: Highest seasonal (6-month) average of 8-hour daily maximum ozone concentrations across 15 months (Jan-Mar)

Using CESM2 SSP2-4.5 ensemble 1 as an example

In [ ]:
import os
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from utils.utils import autosize_figure, land_filter, get_scenario_config

In [ ]:
def process_osdma8_bc_for_plotting(model, scenario, years, ens_num):
    # === Path config ===
    OSDMA8_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"

    dates = f"{years.start}-{years.stop - 1}"

    file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    path = os.path.join(OSDMA8_DIR, file)
    da = xr.open_dataarray(path)

    # Calculate end of scenario mean (10 years)
    end_idx = da.sizes["year"]
    # Select the last 10 years
    da_last10 = da.isel(year=slice(end_idx - 10, end_idx))

    # Get the first and last year values
    start_time = da_last10.year[0].item()
    end_time = da_last10.year[-1].item()

    # Calculate temporal mean
    da_mean = da_last10.mean("year")
    return da_mean, start_time, end_time

In [ ]:
def plot_osdma8_bc(da, model, scenario, ens_num, start_year, end_year, SAVE_DIR, land=False):
    # Create figure and layout
    fig = plt.figure(figsize=autosize_figure(1, 1))
    gs = GridSpec(2, 1, height_ratios=[15, 1])

    # Map projection and display
    projection = ccrs.Robinson()
    crs = ccrs.PlateCarree()

    country_borders = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none')

    cmap = plt.get_cmap("magma")

    if land is True:
        da = land_filter(da)

    # Plot map
    ax = fig.add_subplot(gs[0, 0], projection=projection)
    cb = da.plot(
        transform=crs,
        cmap=cmap,
        add_colorbar=False,
        subplot_kws={'projection': projection}
    )
    ax.coastlines(resolution="50m", linewidth=0.75)
    ax.add_feature(country_borders, edgecolor='k', linewidth=0.75)
    plt.title(f"Bias Corrected OSDMA8: Seasonal average of 8hr daily max\n {model} {scenario} {start_year}-{end_year}", fontsize=16)

    # Colorbar
    cax = fig.add_subplot(gs[1, 0])
    col_bar = plt.colorbar(cb, cax=cax, orientation='horizontal')
    col_bar.set_label("Ozone (ppb)", fontsize=13)

    plt.tight_layout()

    if land is True:
        out_file = f"OSDMA8_BC_land_{model}_{scenario}_{ens_num:02d}_{start_year}-{end_year}.png"
    else:
        out_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{start_year}-{end_year}.png"
    out_path = os.path.join(SAVE_DIR, out_file)
    plt.savefig(out_path)
    return

In [ ]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
SAVE_DIR = "/glade/u/home/awells/air_quality_project/plotting/ozone/OSDMA8_BC/"

model = "CESM2"
scenario = "SSP245"
ensemble_number = 1

config = get_scenario_config(model, scenario)
years = config["years"]

da, first_year, last_year = process_osdma8_bc_for_plotting(model, scenario, years, ensemble_number)

# Plotting without ocean
plot_osdma8_bc(da, model, scenario, ensemble_number, first_year, last_year, SAVE_DIR, land=True)
# Plotting with ocean
plot_osdma8_bc(da, model, scenario, ensemble_number, first_year, last_year, SAVE_DIR, land=False)